# 🔍 Step 1.6: DINOv2 セグメンテーションヘッド学習

```
FPV画像 (224×224×3)
  ↓ DINOv2 Patch tokens (256パッチ × 384次元)
  ↓ セグメンテーションヘッド
各ピクセルのクラス予測
  → 道路 / 建物 / 木 / 空 / 地面
  ↓
「前方中央が道路か」→ 移動判定に使用
```

## なぜこれが重要か

```
現在の移動判定:
  MAP[r][c] == ROAD  ← マップ配列を直接参照 (チート)

Step1.6 導入後:
  FPV画像 → セグメンテーション → 前方が道路か判断
  → マップ配列に一切依存しない
  → 現実のロボットカメラと同じ判断プロセス
```

## データ生成方針

**シミュレーター内で自動ラベル生成 (アノテーション作業ゼロ)**

```
ランダムなエージェント位置・向きで FPV画像を撮影
  ↓
同じ位置・向きのレイキャストで各ピクセルのセルタイプを取得
  → マップ配列が「答え」として自動付与される
  ↓
数千枚のセグメンテーションデータが数分で生成される
```

## セグメンテーションクラス (5クラス)

| ID | クラス | 説明 |
|----|--------|------|
| 0 | sky | 空 (障害物なし方向の上半分) |
| 1 | ground | 地面 (障害物なし方向の下半分) |
| 2 | road_wall | 道路が続く方向 (柱なし) |
| 3 | building | 建物の壁 |
| 4 | tree | 木 |

In [ ]:
# ============================================================
# セル 1: インストール
# ============================================================
!pip install torch torchvision onnx onnxscript onnxruntime numpy -q
print('✓ Install complete')

In [ ]:
# ============================================================
# セル 2: インポート & 設定
# ============================================================
import torch, torch.nn as nn, onnx
import numpy as np
import math, os, json, random, time
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib, matplotlib.patches as mpatches
matplotlib.rcParams.update({
    'figure.facecolor':'#0a0c10','axes.facecolor':'#12161e',
    'text.color':'#c8d8e8',
})
import torchvision.transforms as T

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/mesa_persona'
ONNX_DIR = '/content/drive/MyDrive/mesa_persona_onnx'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(ONNX_DIR, exist_ok=True)

# マップ定数 (step2 と同じ)
OTHER=0; ROAD=1; BUILDING=2; TREE=3
GRID=30; CELL_SIZE=1.0

# FPV設定 (step2 と完全一致)
IMG_W   = 224
IMG_H   = 224
IMG_CH  = 3
FOV     = math.radians(60.0)
RAY_MAX = 8.0
RAY_STEP= 0.1

# セグメンテーションクラス (5クラス)
SEG_CLASSES = ['sky', 'ground', 'open', 'building', 'tree']
N_SEG = len(SEG_CLASSES)
SEG_COLORS = {
    0: (135,206,235),  # sky     水色
    1: ( 26, 40, 32),  # ground  暗緑
    2: ( 80, 80, 80),  # open    グレー (道路方向)
    3: (196, 32, 32),  # building 赤
    4: ( 35,104, 40),  # tree    緑
}

# DINOv2設定
DINO_MODEL = 'dinov2_vits14'
DINO_DIM   = 384
PATCH_SIZE = 14   # ViT-S/14 のパッチサイズ
N_PATCHES  = (IMG_W // PATCH_SIZE) ** 2  # 16×16 = 256

print(f'IMG: {IMG_W}×{IMG_H}  FOV: {math.degrees(FOV):.0f}°')
print(f'DINOv2: {DINO_MODEL}  patches: {N_PATCHES}  dim: {DINO_DIM}')
print(f'セグクラス数: {N_SEG} ({SEG_CLASSES})')

In [ ]:
# ============================================================
# セル 3: マップ生成 (step2 と同じ有機的マップ)
# ============================================================
from collections import deque

def make_map_organic(size=GRID, seed=None):
    rng  = random.Random(seed)
    grid = np.full((size,size), OTHER, dtype=np.int32)
    step = 4
    rows = list(range(0,size,step))
    cols = list(range(0,size,step))
    for r in rows: grid[r,:] = ROAD
    for c in cols: grid[:,c] = ROAD
    for ri in range(len(rows)-1):
        for ci in range(len(cols)-1):
            r0,r1 = rows[ri]+1,rows[ri+1]
            c0,c1 = cols[ci]+1,cols[ci+1]
            cells = [(r,c) for r in range(r0,r1) for c in range(c0,c1)]
            if not cells: continue
            b = rng.choice(cells); grid[b[0],b[1]] = BUILDING
            for r,c in cells:
                if (r,c)==b: continue
                v=rng.random()
                if v<.25: grid[r,c]=TREE
                elif v<.45: grid[r,c]=BUILDING
    for r in rows: grid[r,:]=ROAD
    for c in cols: grid[:,c]=ROAD

    # 道路削除 (BFS接続保証)
    row_set=set(rows); col_set=set(cols)
    def is_inter(r,c): return r in row_set and c in col_set
    cands=[(r,c) for r in range(size) for c in range(size)
           if grid[r,c]==ROAD and not is_inter(r,c)]
    rng.shuffle(cands)
    def road_ok(g):
        start=next(((r,c) for r in range(size) for c in range(size) if g[r,c]==ROAD),None)
        if not start: return True
        vis=set([start]); q=deque([start])
        while q:
            r,c=q.popleft()
            for dr,dc in[(-1,0),(1,0),(0,-1),(0,1)]:
                nr,nc=r+dr,c+dc
                if 0<=nr<size and 0<=nc<size and (nr,nc) not in vis and g[nr,nc]==ROAD:
                    vis.add((nr,nc)); q.append((nr,nc))
        return all(g[r,c]!=ROAD or (r,c) in vis for r in range(size) for c in range(size))
    ratio=0.25+rng.random()*0.25
    mx=int(len(cands)*ratio); removed=0
    for r,c in cands:
        if removed>=mx: break
        grid[r,c]=OTHER
        if road_ok(grid): grid[r,c]=TREE if rng.random()<0.4 else OTHER; removed+=1
        else: grid[r,c]=ROAD
    return grid

# テスト表示
test_map = make_map_organic(seed=42)
cmap = mcolors.ListedColormap(['#1a3a6a','#d0d0c8','#c42020','#256b28'])
fig,ax=plt.subplots(figsize=(4,4))
ax.imshow(test_map,cmap=cmap,vmin=0,vmax=3,origin='upper')
ax.set_title('有機的マップ (seed=42)',color='#c8d8e8'); ax.axis('off')
plt.tight_layout(); plt.show()
print(f'✓ マップ生成OK  ROAD:{(test_map==ROAD).sum()}  BUILDING:{(test_map==BUILDING).sum()}')

In [ ]:
# ============================================================
# セル 4: FPV画像 + セグメンテーションラベルの自動生成
#
# レイキャスト1本ごとに:
#   ① ヒットしたセルタイプ → 建物/木の柱の色を決定
#   ② ヒットしたセルタイプ → ピクセルのセグクラスを決定
#      柱部分:   BUILDING→3, TREE→4
#      空:       0 (sky)
#      地面:     1 (ground)
#      道路方向: 2 (open, 柱なし)
# ============================================================

# FPVカラーパレット (step2 の CELL_RGB と同じ)
CELL_RGB = np.array([
    [ 12,  30,  74],  # OTHER
    [ 80,  80,  80],  # ROAD
    [196,  32,  32],  # BUILDING
    [ 35, 104,  40],  # TREE
], dtype=np.float32) / 255.0
SKY_RGB   = np.array([ 6, 12, 20], dtype=np.float32) / 255.0
FLOOR_RGB = np.array([26, 40, 32], dtype=np.float32) / 255.0

def render_fpv_and_seg(x, y, th, map_np):
    """
    1エージェント分のFPV画像とセグメンテーションマスクを生成。
    x,y: 連続座標  th: 向き(rad)
    returns:
      img: (H, W, 3) uint8  FPV画像
      seg: (H, W)    int32  セグクラス
    """
    H, W = IMG_H, IMG_W
    img  = np.zeros((H, W, 3), dtype=np.float32)
    seg  = np.zeros((H, W),    dtype=np.int32)

    for xi in range(W):
        ray_angle = th + FOV * (xi/(W-1) - 0.5)
        rdx = math.cos(ray_angle)
        rdy = math.sin(ray_angle)

        hit_type = -1; hit_dist = RAY_MAX
        d = RAY_STEP
        while d < RAY_MAX:
            nx = x + rdx*d; ny = y + rdy*d
            r = int(nx); c = int(ny)
            if r<0 or r>=GRID or c<0 or c>=GRID:
                hit_type=OTHER; hit_dist=d; break
            ct = map_np[r,c]
            if ct != ROAD:
                hit_type=ct; hit_dist=d; break
            d += RAY_STEP

        # 柱の高さ
        col_h = int(min(H*1.5/max(hit_dist,0.1), H)) if hit_type>=0 else 0
        y0 = (H - col_h)//2
        y1 = y0 + col_h
        bright = max(0.15, 1.0 - hit_dist/RAY_MAX) if hit_type>=0 else 0

        # 壁の色
        if hit_type>=0:
            wall_rgb = CELL_RGB[hit_type] * bright
        else:
            wall_rgb = None

        for yi in range(H):
            if yi < H//2:
                img[yi,xi] = SKY_RGB; seg[yi,xi] = 0   # sky
            else:
                img[yi,xi] = FLOOR_RGB; seg[yi,xi] = 1  # ground

            if hit_type >= 0 and y0 <= yi < y1:
                img[yi,xi] = wall_rgb
                # セグクラス: BUILDING→3, TREE→4, OTHER→2
                if hit_type == BUILDING: seg[yi,xi] = 3
                elif hit_type == TREE:   seg[yi,xi] = 4
                else:                    seg[yi,xi] = 2
            elif hit_type < 0:
                # 道路が続く方向 → 柱なし → open
                # 空・地面の上に open フラグを重ねる
                seg[yi,xi] = 2   # open (道路方向)

    return (img*255).clip(0,255).astype(np.uint8), seg


# ── テスト描画 ──
test_bcells = [(r,c) for r in range(GRID) for c in range(GRID) if test_map[r,c]==ROAD]
sx,sy = test_bcells[0]; sx+=0.5; sy+=0.5
test_img, test_seg = render_fpv_and_seg(sx, sy, 0.0, test_map)

seg_vis = np.zeros((*test_seg.shape, 3), dtype=np.uint8)
for cls_id, rgb in SEG_COLORS.items():
    seg_vis[test_seg==cls_id] = rgb

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(10,4))
fig.patch.set_facecolor('#0a0c10')
ax1.imshow(test_img); ax1.set_title('FPV画像',color='#c8d8e8'); ax1.axis('off')
ax2.imshow(seg_vis);  ax2.set_title('セグマスク',color='#c8d8e8'); ax2.axis('off')
patches=[mpatches.Patch(color=np.array(c)/255,label=SEG_CLASSES[i])
         for i,c in SEG_COLORS.items()]
ax2.legend(handles=patches,loc='lower right',fontsize=8,
           facecolor='#0a0c10',labelcolor='#c8d8e8')
plt.tight_layout(); plt.show()
print('✓ FPV画像 + セグマスク生成OK')

In [ ]:
# ============================================================
# セル 5: 大量データ生成
# ============================================================
N_SAMPLES = 5000   # 生成枚数 (増やすほど精度向上)
SAVE_EVERY = 500

data_dir = Path(SAVE_DIR) / 'seg_data'
data_dir.mkdir(exist_ok=True)

# 既存データを確認
existing = list(data_dir.glob('*.npz'))
print(f'既存データ: {len(existing)}枚 / 目標: {N_SAMPLES}枚')

imgs_buf = []
segs_buf = []
total_saved = len(existing)

print('データ生成中...')
t0 = time.time()

for i in range(N_SAMPLES - len(existing)):
    # ランダムなマップ・位置・向きで生成
    map_np = make_map_organic(seed=random.randint(0, 99999))
    road_cells = [(r,c) for r in range(GRID) for c in range(GRID) if map_np[r,c]==ROAD]
    if not road_cells: continue

    r,c  = random.choice(road_cells)
    x    = r + random.uniform(0.1, 0.9)
    y    = c + random.uniform(0.1, 0.9)
    th   = random.uniform(0, math.pi*2)

    img, seg = render_fpv_and_seg(x, y, th, map_np)
    imgs_buf.append(img)
    segs_buf.append(seg)

    # バッファが溜まったら保存
    if len(imgs_buf) >= SAVE_EVERY:
        np.savez_compressed(
            data_dir / f'batch_{total_saved:05d}.npz',
            imgs=np.stack(imgs_buf),
            segs=np.stack(segs_buf))
        total_saved += len(imgs_buf)
        imgs_buf = []; segs_buf = []
        elapsed = time.time()-t0
        print(f'  {total_saved}/{N_SAMPLES}枚  '
              f'({elapsed:.0f}s  {total_saved/elapsed:.0f}枚/s)')

# 残りを保存
if imgs_buf:
    np.savez_compressed(
        data_dir / f'batch_{total_saved:05d}.npz',
        imgs=np.stack(imgs_buf), segs=np.stack(segs_buf))
    total_saved += len(imgs_buf)

print(f'\n✓ 生成完了: {total_saved}枚  ({time.time()-t0:.0f}s)')

In [ ]:
# ============================================================
# セル 6: DINOv2 ロード & セグメンテーションヘッド定義
# ============================================================

print('DINOv2 をロード中...')
dino = torch.hub.load('facebookresearch/dinov2', DINO_MODEL, pretrained=True)
dino = dino.to(DEVICE).eval()
for p in dino.parameters(): p.requires_grad = False
print(f'✓ DINOv2 ({sum(p.numel() for p in dino.parameters()):,} params, frozen)')

# DINOv2 の Patch tokens を取得するラッパー
def get_patch_tokens(imgs_tensor):
    """
    imgs_tensor: (N, 3, H, W)  [0,1] 正規化済み
    returns: patch_tokens (N, N_PATCHES, DINO_DIM)
    """
    with torch.no_grad():
        out = dino.forward_features(imgs_tensor)
        # DINOv2 の forward_features は dict を返す
        # 'x_norm_patchtokens': (N, N_PATCHES, dim)
        patch = out['x_norm_patchtokens']   # (N, 256, 384)
    return patch

# ImageNet 正規化
img_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# ── セグメンテーションヘッド ──
# Patch token (256×384) → 各パッチのクラス (256×N_SEG)
# → upsample → 各ピクセルのクラス (224×224×N_SEG)

class SegHead(nn.Module):
    """
    DINOv2 の Patch tokens → セグメンテーションマップ

    DINOv2 ViT-S/14: パッチサイズ14×14, 画像224×224 → 16×16=256パッチ
    各パッチを分類してから 224×224 にアップサンプル
    """
    def __init__(self, in_dim=DINO_DIM, n_classes=N_SEG):
        super().__init__()
        P = IMG_W // PATCH_SIZE   # 16
        self.P = P
        # パッチレベル分類器
        self.patch_cls = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Linear(128, n_classes),
        )
        # アップサンプラー (16×16 → 224×224)
        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(n_classes, 64, 4, stride=2, padding=1),  # 32×32
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),         # 64×64
            nn.ReLU(),
            nn.ConvTranspose2d(32, n_classes, 4, stride=2, padding=1),  # 128×128
            nn.Upsample(size=(IMG_H, IMG_W), mode='bilinear', align_corners=False),
        )

    def forward(self, patch_tokens):
        """
        patch_tokens: (N, N_PATCHES, DINO_DIM)
        returns: logits (N, N_SEG, H, W)
        """
        N = patch_tokens.shape[0]
        P = self.P
        # パッチを分類
        patch_logits = self.patch_cls(patch_tokens)   # (N, 256, N_SEG)
        # 2D に reshape
        patch_2d = patch_logits.reshape(N, P, P, N_SEG)  # (N, 16, 16, C)
        patch_2d = patch_2d.permute(0, 3, 1, 2)           # (N, C, 16, 16)
        # アップサンプル
        return self.upsample(patch_2d)  # (N, C, 224, 224)


seg_head = SegHead().to(DEVICE)
n_params = sum(p.numel() for p in seg_head.parameters())
print(f'SegHead: {n_params:,} params (これだけ学習)')

# テスト
dummy_patch = torch.randn(2, N_PATCHES, DINO_DIM, device=DEVICE)
dummy_out   = seg_head(dummy_patch)
print(f'forward OK: {dummy_patch.shape} → {dummy_out.shape}')
del dummy_patch, dummy_out

In [ ]:
# ============================================================
# セル 7: セグメンテーションヘッドの学習
# ============================================================
from torch.utils.data import Dataset, DataLoader

class SegDataset(Dataset):
    def __init__(self, data_dir, transform):
        self.files = sorted(Path(data_dir).glob('*.npz'))
        self.transform = transform
        # 全データをロードしてキャッシュ
        imgs_list, segs_list = [], []
        for f in self.files:
            d = np.load(f)
            imgs_list.append(d['imgs'])  # (B, H, W, 3) uint8
            segs_list.append(d['segs'])  # (B, H, W)    int32
        self.imgs = np.concatenate(imgs_list, axis=0)
        self.segs = np.concatenate(segs_list, axis=0)
        print(f'Dataset: {len(self.imgs)}枚')

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img = self.transform(self.imgs[idx])        # (3, H, W)
        seg = torch.tensor(self.segs[idx], dtype=torch.long)  # (H, W)
        return img, seg

# データセット作成
dataset    = SegDataset(data_dir, img_transform)
n_train    = int(len(dataset)*0.85)
n_val      = len(dataset) - n_train
train_ds, val_ds = torch.utils.data.random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

BATCH_SIZE = 16
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Train:{len(train_ds)}  Val:{len(val_ds)}  Batch:{BATCH_SIZE}')

# 学習
optimizer = torch.optim.Adam(seg_head.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

N_EPOCHS = 20
best_miou = 0.0
train_losses, val_mious = [], []

from IPython.display import clear_output

for epoch in range(N_EPOCHS):
    # Train
    seg_head.train()
    ep_loss = 0
    for imgs, segs in train_dl:
        imgs = imgs.to(DEVICE); segs = segs.to(DEVICE)
        with torch.no_grad():
            patch_tokens = get_patch_tokens(imgs)
        logits = seg_head(patch_tokens)   # (B, C, H, W)
        loss   = criterion(logits, segs)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        ep_loss += loss.item()
    scheduler.step()
    ep_loss /= len(train_dl)

    # Validation mIoU
    seg_head.eval()
    iou_sum = np.zeros(N_SEG)
    iou_cnt = np.zeros(N_SEG)
    with torch.no_grad():
        for imgs, segs in val_dl:
            imgs = imgs.to(DEVICE)
            patch_tokens = get_patch_tokens(imgs)
            pred = seg_head(patch_tokens).argmax(dim=1).cpu().numpy()  # (B, H, W)
            true = segs.numpy()
            for cls in range(N_SEG):
                inter = ((pred==cls)&(true==cls)).sum()
                union = ((pred==cls)|(true==cls)).sum()
                if union > 0:
                    iou_sum[cls] += inter/union
                    iou_cnt[cls] += 1
    miou = np.nanmean(iou_sum / np.maximum(iou_cnt, 1))
    train_losses.append(ep_loss)
    val_mious.append(miou)

    if miou > best_miou:
        best_miou = miou
        best_state = {k:v.clone() for k,v in seg_head.state_dict().items()}

    if (epoch+1)%5==0 or epoch==0:
        clear_output(wait=True)
        fig,(ax1,ax2)=plt.subplots(1,2,figsize=(10,3))
        fig.patch.set_facecolor('#0a0c10')
        ax1.plot(train_losses,color='#ff5050',lw=1.5); ax1.set_title('Train Loss',color='#c8d8e8')
        ax2.plot(val_mious,color='#00ddb4',lw=1.5); ax2.set_title('Val mIoU',color='#c8d8e8')
        ax2.set_ylim(0,1)
        for ax in[ax1,ax2]: ax.grid(color='#1e2530',lw=0.5); ax.spines[:].set_color('#1e2530')
        plt.suptitle(f'Epoch {epoch+1}/{N_EPOCHS}  mIoU:{miou:.3f}  Best:{best_miou:.3f}',
                     color='#00ddb4'); plt.tight_layout(); plt.show()
        cls_ious=[f'{SEG_CLASSES[i]}:{iou_sum[i]/max(iou_cnt[i],1):.2f}' for i in range(N_SEG)]
        print(' | '.join(cls_ious))

seg_head.load_state_dict(best_state)
print(f'\n✓ 学習完了  Best mIoU: {best_miou:.3f}')

In [ ]:
# ============================================================
# セル 8: 予測結果の可視化
# ============================================================
seg_head.eval()
test_map2 = make_map_organic(seed=999)
road_cells2 = [(r,c) for r in range(GRID) for c in range(GRID) if test_map2[r,c]==ROAD]

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.patch.set_facecolor('#0a0c10')

for ax_row in range(3):
    r,c = random.choice(road_cells2)
    x = r+0.5; y = c+0.5; th = random.uniform(0, math.pi*2)
    img_np, seg_true = render_fpv_and_seg(x, y, th, test_map2)

    img_t = img_transform(img_np).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        patch = get_patch_tokens(img_t)
        pred  = seg_head(patch).argmax(dim=1)[0].cpu().numpy()

    def to_rgb(seg_arr):
        out = np.zeros((*seg_arr.shape, 3), dtype=np.uint8)
        for cid,rgb in SEG_COLORS.items(): out[seg_arr==cid]=rgb
        return out

    axes[ax_row][0].imshow(img_np); axes[ax_row][0].set_title('FPV画像',color='#c8d8e8',fontsize=8)
    axes[ax_row][1].imshow(to_rgb(seg_true)); axes[ax_row][1].set_title('正解',color='#c8d8e8',fontsize=8)
    axes[ax_row][2].imshow(to_rgb(pred));     axes[ax_row][2].set_title('予測',color='#00ddb4',fontsize=8)
    for ax in axes[ax_row]: ax.axis('off')

plt.suptitle(f'セグメンテーション結果 (mIoU={best_miou:.3f})',
             color='#00ddb4',fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# セル 9: ONNX エクスポート
# ============================================================
seg_head.eval().cpu()

onnx_path = f'{ONNX_DIR}/seg_head.onnx'
dummy = torch.zeros(1, N_PATCHES, DINO_DIM)

with torch.no_grad():
    torch.onnx.export(
        seg_head, dummy, onnx_path,
        input_names=['patch_tokens'],
        output_names=['seg_logits'],
        dynamic_axes={
            'patch_tokens': {0:'batch'},
            'seg_logits':   {0:'batch'},
        },
        opset_version=17,
    )

onnx_model = onnx.load(onnx_path)
onnx.save(onnx_model, onnx_path, save_as_external_data=False)
print(f'✓ seg_head.onnx ({os.path.getsize(onnx_path)/1e6:.2f}MB)')

# 推論テスト
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
dummy_np = np.zeros((1, N_PATCHES, DINO_DIM), dtype=np.float32)
out = sess.run(None, {'patch_tokens': dummy_np})[0]
print(f'✓ 推論テスト OK  出力shape: {out.shape}  (N, {N_SEG}, {IMG_H}, {IMG_W})')

# メタデータ
meta = {
    'classes':      SEG_CLASSES,
    'n_classes':    N_SEG,
    'dino_model':   DINO_MODEL,
    'dino_dim':     DINO_DIM,
    'n_patches':    N_PATCHES,
    'patch_size':   PATCH_SIZE,
    'img_w': IMG_W, 'img_h': IMG_H,
    'input_name':   'patch_tokens',
    'output_name':  'seg_logits',
    'best_miou':    float(best_miou),
    'seg_colors':   {str(k):list(v) for k,v in SEG_COLORS.items()},
    'open_class_id': 2,   # 「道路方向=通行可」のクラスID
}
meta_path = onnx_path.replace('.onnx', '_meta.json')
with open(meta_path,'w') as f:
    json.dump(meta, f, indent=2)
print(f'✓ seg_head_meta.json 保存')

print()
print('='*50)
print('✅ Step 1.6 完了!')
print(f'   保存先: {ONNX_DIR}')
print(f'   - seg_head.onnx')
print(f'   - seg_head_meta.json')
print()
print('次のステップ:')
print('  step2_persona_train.ipynb のセル7を更新すると')
print('  セグメンテーション結果を移動判定に使用できます')
print(f'  open_class_id={meta["open_class_id"]} (前方中央がこのクラス → 前進可能)')